# Session based evaluation

Typically in search systems we aggregate many sessions to get a label for a query / document pair. For example the total number of clicks divided by the number of sessions a product appears in. We might call that "click-thru rate" - a number from 0 to 1 that describes a documents relevance from 0 to 1.

In this notebook, we demonstrate an alternative approach *replaying the sessions themselves*. This can be useful for many sparse tail queries (like natural language). Sampling actual sessions has advantages:

* We can get a good signal on tail queries - aggregating many queries with low amounts of traffic - to give us a picture of that population.
* We don't need to worry about weighing queries by their frequency. By sampling a random session, we just give equal weight with every other session.

Its major disadvantage: its harder *make a claim about a single query*. 

Session-based evaluation comes closer to an "A/B test simulation", while traditional search evaluation comes closer to "a way to debug individual queries reliably"

## This notebook's code

Below we have a set of simulated historical sessions from the retrotech e-commerce dataset. Here a session means "user issued query->got list of results->clicked / didnt click something." We could replace "click" with any number of events (hovers, conversions, etc).

For each session below, we replay the query, and observe whether the clicked result moved up or down. We hope to hove it UP on average.

We measure this with [mean reciprocal rank](https://en.wikipedia.org/wiki/Mean_reciprocal_rank)

In [1]:
from aips import *
from ltr.judgments import Judgment
from ltr.sdbn_functions import *
import aips.indexer
from aips import get_engine
pandas.set_option("display.width", 120)
aips.indexer.pull_github_repository("signals")
aips.indexer.build_collection(get_engine(), "products")

%matplotlib inline

Pulling existing repo [data/repositories/retrotech]


# Load sessions

Here we have 55,000 unique sessions we could replay.

In [2]:
sessions = all_sessions()
sessions

,sess_id,query,rank,doc_id,clicked
0,50002,blue ray,0.0,600603141003,True
1,50002,blue ray,1.0,827396513927,False
2,50002,blue ray,2.0,24543672067,False
3,50002,blue ray,3.0,719192580374,False
4,50002,blue ray,4.0,885170033412,True
...,...,...,...,...,...
74995,5001,transformers dark of the moon,10.0,47875841369,False
74996,5001,transformers dark of the moon,11.0,97363560449,False
74997,5001,transformers dark of the moon,12.0,93624956037,False
74998,5001,transformers dark of the moon,13.0,97363532149,False


## Observe we have multiple sessions with same query

As our central unit of evaluation is a *session* NOT a *query*, we expect different sessions with the same query. 

We also expect them to have different 'right' answers.

That's ok! In a way, BOTH sessions are correct. And a good ranker will move each session's right answer close to the top.

In [3]:
sessions[sessions['sess_id'] == 5004][:10]

,sess_id,query,rank,doc_id,clicked
60,5004,iphone,0.0,885909510344,True
61,5004,iphone,1.0,884720011757,False
62,5004,iphone,2.0,705105070803,False
63,5004,iphone,3.0,885909343874,False
64,5004,iphone,4.0,722868816622,False
65,5004,iphone,5.0,885909394494,False
66,5004,iphone,6.0,885909420438,False
67,5004,iphone,7.0,885909459834,False
68,5004,iphone,8.0,885909538027,False
69,5004,iphone,9.0,97855069917,True


In [4]:
sessions[sessions['sess_id'] == 5003][:10]

,sess_id,query,rank,doc_id,clicked
30,5003,iphone,0.0,885909510344,False
31,5003,iphone,1.0,843404064434,False
32,5003,iphone,2.0,885909459858,False
33,5003,iphone,3.0,885909343874,False
34,5003,iphone,4.0,722868816622,False
35,5003,iphone,5.0,705105070803,True
36,5003,iphone,6.0,685387305636,False
37,5003,iphone,7.0,885909459834,False
38,5003,iphone,8.0,885909538027,False
39,5003,iphone,9.0,97855069917,False


## The search we want to evaluate

Here we have a naive search. Just searching in Solr the name and description. No weight is given to either.

In [7]:
import pandas as pd
import requests
import numpy as np

def search_baseline(keywords: str, rows=10):
    params = {'q':  keywords,
              'defType': 'edismax',
              'qf': 'name long_description',
              'wt': 'json',
              'echoParams': 'all',
              'rows': rows}
    resp = requests.get('http://aips-solr:8983/solr/products/select', params=params)
    json_resp = resp.json()
    docs = json_resp['response']['docs']
    doc_df = pd.DataFrame(docs)
    if len(docs) > 0:
        doc_df = doc_df.rename(columns={'upc': 'doc_id'})
        doc_df['rank'] = np.arange(len(doc_df)) + 1
        doc_df['doc_id'] = doc_df['doc_id'].astype(int)
    return doc_df

results_df = search_baseline("transformers")
results_df

,doc_id,name,name_ngram,name_omit_norms,name_txt_en_split,manufacturer,short_description,long_description,id,_version_,rank
0,708056579746,Nintendo - Transformers 3 Stylus 2-Pack,Nintendo - Transformers 3 Stylus 2-Pack,Nintendo - Transformers 3 Stylus 2-Pack,Nintendo - Transformers 3 Stylus 2-Pack,Nintendo,Take on your adventures with Bumblebee and Opt...,Transform your stylus into two of the stars fr...,7c98f35f-1639-42df-9f39-a0aeff0cfdb5,1849071180397412408,1
1,708056579739,Nintendo - Transformers 3 Cybertanium Case,Nintendo - Transformers 3 Cybertanium Case,Nintendo - Transformers 3 Cybertanium Case,Nintendo - Transformers 3 Cybertanium Case,Nintendo,Protect your game system in true Transformers ...,You love your handheld gaming system and you l...,d3f8174e-8035-427b-9bfb-e9dbbf1561e9,1849071180397412401,2
2,32429037763,Transformers - DVD,Transformers - DVD,Transformers - DVD,Transformers - DVD,,,,4e49dba5-ca85-4396-bff0-fa9e3cfebe13,1849071180843057234,3
3,47875332911,Transformers: Revenge of the Fallen - Windows,Transformers: Revenge of the Fallen - Windows,Transformers: Revenge of the Fallen - Windows,Transformers: Revenge of the Fallen - Windows,Activision,Will you become an Autobot or a Decepticon?,SynopsisBrace yourself for awesome robotic thr...,68f8760b-1515-4ff3-bbc4-91ae175035d1,1849071181337985043,4
4,879862003500,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer,Infrared technology; 3500 dpi resolution; 1000...,"This mouse features a 3500 dpi resolution, 3.5...",31c8d8d4-4524-42ff-b642-96b76b2f26f7,1849071180310380561,5
5,879862003531,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer,Infrared technology; 3500 dpi resolution; 1000...,Get your game on with this mouse that features...,5d1995c4-467a-452e-ac45-6bfc96ce58a0,1849071180310380556,6
6,812726014915,Mimobot - Optimus Prime 4GB USB 2.0 Flash Drive,Mimobot - Optimus Prime 4GB USB 2.0 Flash Drive,Mimobot - Optimus Prime 4GB USB 2.0 Flash Drive,Mimobot - Optimus Prime 4GB USB 2.0 Flash Drive,Mimobot,Compatible with PC and Mac; 4GB storage capaci...,This USB 2.0 flash drive features a Transforme...,ef0591ac-7ae9-4cda-a187-c9ad4c0d7d7e,1849071180901777439,7
7,879862003524,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer,Infrared technology; 3500 dpi resolution; 1000...,Play your favorite computer games with this mo...,25102f3b-9a72-41a9-a653-c946ba09e1b9,1849071180310380553,8
8,93624995012,Transformers - Original Soundtrack - CD,Transformers - Original Soundtrack - CD,Transformers - Original Soundtrack - CD,Transformers - Original Soundtrack - CD,Warner Bros.,,,c18f545d-6200-4926-9a77-a44111fe95f6,1849071180435161139,9
9,879862003494,Razer - Vespula Transformers 3 Collector's Edi...,Razer - Vespula Transformers 3 Collector's Edi...,Razer - Vespula Transformers 3 Collector's Edi...,Razer - Vespula Transformers 3 Collector's Edi...,Razer,Dual-sided design; optimized for speed or cont...,This mouse mat features a dual-sided design wi...,6bff1541-cfa8-4d6d-b4f8-84ae469b7add,1849071180310380558,10


## Eval with 1000 sessions

Take NUM_SESSIONS random sessions and replay for eval.

* Replay each sessions query
* Note the rank of the click from previous sessions
* Append to dictionary of all session results

In [8]:
import numpy as np

NUM_SESSIONS = 1000
np.random.seed(42)

def best_click(session):
    if session['clicked'].sum():
        clicked_rank = session[session['clicked']]['rank'].min()
        return session[session['rank'] == clicked_rank]

    
def replay(sessions, num_sessions, search_fn):
    sess_ids = np.random.choice(sessions['sess_id'].unique(), NUM_SESSIONS)

    all_results = {}

    for sess_id in sess_ids:
        session = sessions[sessions['sess_id'] == sess_id].copy()
        query = session['query'].iloc[0]
        clicked_row = best_click(session)
        if clicked_row is not None:
            clicked_doc_id = clicked_row['doc_id'].iloc[0]
            
            # In reality, for offline replay, you might
            # not 
            results = search_fn(query)
            if len(results):
                results['clicked'] = results['doc_id'] == clicked_doc_id
                results['sess_id'] = sess_id
            else:
                # 0 search results
                results = pd.DataFrame(columns=['clicked', 'sess_id', 'doc_id', 'name',
                                                'long_description', 'short_description'])
            all_results[sess_id] = (query, results)
    return all_results

all_results = replay(sessions, NUM_SESSIONS, search_baseline)

## Examine replay results

We see the replay results for a session. Observe the rank of a click - click from when we gathered this session. Note the session selected might NOT have a click.

In [9]:
some_sess_id = list(all_results.keys())[10]
query, sess_results = all_results[some_sess_id]
print(query)
sess_results

star trek


,doc_id,name,name_ngram,name_omit_norms,name_txt_en_split,manufacturer,short_description,long_description,id,_version_,rank,clicked,sess_id
0,742725280410,Star Trek Online Collector's Edition - Windows,Star Trek Online Collector's Edition - Windows,Star Trek Online Collector's Edition - Windows,Star Trek Online Collector's Edition - Windows,Atari,Blast off into interstellar gameplay and aweso...,SynopsisThis Collector's Edition provides incr...,09a6ca12-05e1-4459-b77c-460b6c2cbe5c,1849071180222300161,1,False,47193
1,97361237244,Star Trek: Captains Log - DVD,Star Trek: Captains Log - DVD,Star Trek: Captains Log - DVD,Star Trek: Captains Log - DVD,,,,c37e0d70-612a-4c12-a900-c0468857cb74,1849071180499124241,2,False,47193
2,97361301747,Star Trek: Fan Collectives - DVD,Star Trek: Fan Collectives - DVD,Star Trek: Fan Collectives - DVD,Star Trek: Fan Collectives - DVD,,,,2e8039eb-a6f3-4ab0-831c-057d7b2f56cc,1849071180500172801,3,False,47193
3,738572128920,Music Of Star Trek - O.S.T. - CD,Music Of Star Trek - O.S.T. - CD,Music Of Star Trek - O.S.T. - CD,Music Of Star Trek - O.S.T. - CD,Silva America,,,dbd7cbfc-1213-40d2-a3f3-cfe6f74fa583,1849071180783288362,4,False,47193
4,738572115524,Star Trek Album - Original Soundtrack - CD,Star Trek Album - Original Soundtrack - CD,Star Trek Album - Original Soundtrack - CD,Star Trek Album - Original Soundtrack - CD,Silva America,,,d5ba87fd-62be-4808-a233-6b4f3743b6b1,1849071180478152782,5,False,47193
5,30206696622,Star Trek (Score) - Original Soundtrack - CD,Star Trek (Score) - Original Soundtrack - CD,Star Trek (Score) - Original Soundtrack - CD,Star Trek (Score) - Original Soundtrack - CD,Var&#xBF;se Sarabande (USA),,,24164fe4-e171-4656-81a1-0bf37fdc31ea,1849071181355810830,6,False,47193
6,97360709742,Star Trek: Fan Collective - Klingon - DVD,Star Trek: Fan Collective - Klingon - DVD,Star Trek: Fan Collective - Klingon - DVD,Star Trek: Fan Collective - Klingon - DVD,,,,1806ed0a-e82f-47ec-8b9d-b8831be56e67,1849071181105201172,7,False,47193
7,791149900183,Star Trek: The Complete Comic Book Collection ...,Star Trek: The Complete Comic Book Collection ...,Star Trek: The Complete Comic Book Collection ...,Star Trek: The Complete Comic Book Collection ...,Gitcorp,Revisit the final frontier,Fans of the Star Trek comic book series will e...,90921890-8ab3-411c-9ca2-eaa4f62ed03f,1849071180375392283,8,False,47193
8,97363485049,Star Trek - Widescreen Dubbed Subtitle AC3 - DVD,Star Trek - Widescreen Dubbed Subtitle AC3 - DVD,Star Trek - Widescreen Dubbed Subtitle AC3 - DVD,Star Trek - Widescreen Dubbed Subtitle AC3 - DVD,,,,2f4fa8f2-d65c-47f9-9fef-ab6fc364686f,1849071180790628383,9,False,47193
9,97360719642,Star Trek: Original Motion Picture Collection ...,Star Trek: Original Motion Picture Collection ...,Star Trek: Original Motion Picture Collection ...,Star Trek: Original Motion Picture Collection ...,,,,cde4c805-8672-427b-93c2-e3e3a7205336,1849071180791676937,10,False,47193


## Compute MRR of naive approach.

We take the click rank of every previous session and compute the reciprocal rank (rr). Take the average to get MRR.

We then output the MRR of this naive approach. If MRR were 1, each sessions clicked UPC would be at the best position possible (rank=1).

In [10]:
def compute_mrr(all_results):

    sum_rrs = 0

    for sess_id, q_session_results in all_results.items():
        query, session_results = q_session_results
        reciprocal_rank = 0
        if len(session_results) > 0:
            clicked_row = best_click(session_results)
            if clicked_row is not None:
                click_rank = clicked_row['rank'].iloc[0]
                reciprocal_rank = 1 / click_rank
        sum_rrs += reciprocal_rank

    mrr = sum_rrs / len(all_results)
    return mrr

compute_mrr(all_results)

0.08569082805536012

## Try a new type of search - high title boost

Note compared to the previous search, we have a title boost of 10 (the `^10`)

Here we implement this approach, below we'll replay all the sessions to see how it impacted MRR.

In [12]:
import pandas as pd
import requests

def search_title_boost(keywords: str, rows=10):
    params = {'q':  keywords,
              'defType': 'edismax',
              'qf': 'name^10 long_description',
              'wt': 'json',
              'echoParams': 'all',
              'rows': rows}
    resp = requests.get('http://aips-solr:8983/solr/products/select', params=params)
    json_resp = resp.json()
    docs = json_resp['response']['docs']
    doc_df = pd.DataFrame(docs)
    if len(docs) > 0:
        doc_df = doc_df.rename(columns={'upc': 'doc_id'})
        doc_df['rank'] = np.arange(len(doc_df)) + 1
        doc_df['doc_id'] = doc_df['doc_id'].astype(int)
    return doc_df

results_df = search_baseline("transformers")
results_df

,doc_id,name,name_ngram,name_omit_norms,name_txt_en_split,manufacturer,short_description,long_description,id,_version_,rank
0,708056579746,Nintendo - Transformers 3 Stylus 2-Pack,Nintendo - Transformers 3 Stylus 2-Pack,Nintendo - Transformers 3 Stylus 2-Pack,Nintendo - Transformers 3 Stylus 2-Pack,Nintendo,Take on your adventures with Bumblebee and Opt...,Transform your stylus into two of the stars fr...,7c98f35f-1639-42df-9f39-a0aeff0cfdb5,1849071180397412408,1
1,708056579739,Nintendo - Transformers 3 Cybertanium Case,Nintendo - Transformers 3 Cybertanium Case,Nintendo - Transformers 3 Cybertanium Case,Nintendo - Transformers 3 Cybertanium Case,Nintendo,Protect your game system in true Transformers ...,You love your handheld gaming system and you l...,d3f8174e-8035-427b-9bfb-e9dbbf1561e9,1849071180397412401,2
2,32429037763,Transformers - DVD,Transformers - DVD,Transformers - DVD,Transformers - DVD,,,,4e49dba5-ca85-4396-bff0-fa9e3cfebe13,1849071180843057234,3
3,47875332911,Transformers: Revenge of the Fallen - Windows,Transformers: Revenge of the Fallen - Windows,Transformers: Revenge of the Fallen - Windows,Transformers: Revenge of the Fallen - Windows,Activision,Will you become an Autobot or a Decepticon?,SynopsisBrace yourself for awesome robotic thr...,68f8760b-1515-4ff3-bbc4-91ae175035d1,1849071181337985043,4
4,879862003500,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer,Infrared technology; 3500 dpi resolution; 1000...,"This mouse features a 3500 dpi resolution, 3.5...",31c8d8d4-4524-42ff-b642-96b76b2f26f7,1849071180310380561,5
5,879862003531,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer,Infrared technology; 3500 dpi resolution; 1000...,Get your game on with this mouse that features...,5d1995c4-467a-452e-ac45-6bfc96ce58a0,1849071180310380556,6
6,812726014915,Mimobot - Optimus Prime 4GB USB 2.0 Flash Drive,Mimobot - Optimus Prime 4GB USB 2.0 Flash Drive,Mimobot - Optimus Prime 4GB USB 2.0 Flash Drive,Mimobot - Optimus Prime 4GB USB 2.0 Flash Drive,Mimobot,Compatible with PC and Mac; 4GB storage capaci...,This USB 2.0 flash drive features a Transforme...,ef0591ac-7ae9-4cda-a187-c9ad4c0d7d7e,1849071180901777439,7
7,879862003524,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer - DeathAdder Transformers 3 Collector's ...,Razer,Infrared technology; 3500 dpi resolution; 1000...,Play your favorite computer games with this mo...,25102f3b-9a72-41a9-a653-c946ba09e1b9,1849071180310380553,8
8,93624995012,Transformers - Original Soundtrack - CD,Transformers - Original Soundtrack - CD,Transformers - Original Soundtrack - CD,Transformers - Original Soundtrack - CD,Warner Bros.,,,c18f545d-6200-4926-9a77-a44111fe95f6,1849071180435161139,9
9,879862003494,Razer - Vespula Transformers 3 Collector's Edi...,Razer - Vespula Transformers 3 Collector's Edi...,Razer - Vespula Transformers 3 Collector's Edi...,Razer - Vespula Transformers 3 Collector's Edi...,Razer,Dual-sided design; optimized for speed or cont...,This mouse mat features a dual-sided design wi...,6bff1541-cfa8-4d6d-b4f8-84ae469b7add,1849071180310380558,10


### Compute MRR of 'title boost' over all sessions

In [13]:
all_results = replay(sessions, NUM_SESSIONS, search_title_boost)
compute_mrr(all_results)

0.11630753968253969

## Points of discussion / experimentation

* Where does this approach fall down? What are the downsides? Consider perhaps we might need to sample many sessions, with many unique queries, to get a complete picture?
* One reason to do this- we don't have enough natural language queries to aggregate many sessions with that query to get a complete picture. But maybe there could be a way of aggregating them by semantic similarity?
* How does position / presentation bias play in here? We obviously won't have many clicks for items towards the bottom or that do not appear. Does that say something about the non-clicked items above the far-down clicked one? Should we somehow use this information?
